In [0]:
%run /Workspace/Users/weyannago@gmail.com/malbank/setup_file

In [0]:
file_path=f"/Volumes/{catalog}/{schema}/{volume}/landing/"
output_path=f"/Volumes/{catalog}/{schema}/{volume}/payment_event/"

In [0]:
# def canonical_payment_schema():
#     schema=StructType([
#         StructField('event_id',StringType(),True),
#         StructField('payment_type',StringType(),True),
#         StructField('customer_id',StringType(),True),
#         StructField('amount',FloatType(),True),
#         StructField('currency',StringType(),True),
#         StructField('status',StringType(),True),
#         StructField('timestamp',DateType(),True),
#     ])
#     return schema

In [0]:
def tranform_bill_payment():
    df=spark.read.option("header", True).option("inferSchema", True).csv(f"{file_path}/bill_payments.csv")
    df=df.withColumnRenamed('bill_ref','event_id')\
        .withColumnRenamed('customerId','customer_id')\
        .withColumnRenamed('bill_amount','amount')\
        .withColumnRenamed('currency_code','currency')\
        .withColumnRenamed('payment_date','timestamp')\
        .withColumnRenamed('success_flag','status')\
        .withColumn('amount',F.col("amount").cast("double"))\
        .withColumn('currency',F.upper(F.col("currency")))\
        .filter(F.col("amount")>0)\
        .withColumn('timestamp',F.col("timestamp").cast("date"))\
        .withColumn('status',F.when(F.col("status")==True,'success').otherwise('failed'))\
        .withColumn('payment_type',F.lit('bill_payment'))
    return df

dfs=tranform_bill_payment()
display(dfs)


In [0]:
def tranform_bill_payment():
    df=spark.read.option("header", True).option("inferSchema", True).csv(f"{file_path}/bill_payments.csv")
    df=df.withColumnRenamed('bill_ref','event_id')\
        .withColumnRenamed('customerId','customer_id')\
        .withColumnRenamed('bill_amount','amount')\
        .withColumnRenamed('currency_code','currency')\
        .withColumnRenamed('payment_date','timestamp')\
        .withColumnRenamed('success_flag','status')\
        .withColumn('amount',F.col("amount").cast("double"))\
        .withColumn('currency',F.upper(F.col("currency")))\
        .filter(F.col("amount")>0)\
        .withColumn('timestamp',F.col("timestamp").cast("date"))\
        .withColumn('status',F.when(F.col("status")==True,'success').otherwise('failed'))\
        .withColumn('payment_type',F.lit('bill_payment'))
    return df

In [0]:
def tranform_card_payment():
    df=spark.read.option("header", True).option("inferSchema", True).csv(f"{file_path}/cards.csv")
    df=df.withColumnRenamed('txn_id','event_id')\
        .withColumnRenamed('card_no','customer_id')\
        .withColumnRenamed('amount_usd','amount')\
        .withColumnRenamed('status','status')\
        .withColumnRenamed('created_at','timestamp')\
        .withColumn('amount',F.col("amount").cast("double"))\
        .withColumn('currency',F.lit('USD'))\
        .withColumn('timestamp',F.col("timestamp").cast("date"))\
        .withColumn('status',F.when(F.col("status")=='approved','success').when(F.col("status")=='pending','pending')\
        .otherwise('failed'))\
        .withColumn('payment_type',F.lit('card_payment'))
    return df




In [0]:
def tranform_transfer_payment():
    df=spark.read.option("header", True).option("inferSchema", True).csv(f"{file_path}/transfers.csv")
    change_columns = {'transfer_id':'event_id','from_account':'customer_id','amount':'amount','currency':'currency','state':'status','created_at':'timestamp'}
    df=df.withColumnsRenamed(change_columns).\
        withColumn('amount',F.col("amount").cast("double"))\
        .withColumn('currency',F.upper(F.col("currency")))\
        .filter(F.col("amount")>0)\
        .withColumn('timestamp',F.col("timestamp").cast("date"))\
        .withColumn('payment_type',F.lit('transfer_payment'))\
        .withColumn('status',F.when(F.col("status")=='completed','success').when(F.col("status")=='processing','pending')\
        .otherwise('failed'))
    return df
dfs=tranform_transfer_payment()
display(dfs)

In [0]:
def raw_to_gold():
    transfer=tranform_transfer_payment()
    card=tranform_card_payment()
    bill=tranform_bill_payment()
    df_merge=transfer.unionByName(card).unionByName(bill)
    df_merge.write.mode("overwrite").parquet(output_path)

raw_to_gold()
unified_df=spark.read.parquet(output_path)
unified_df.createOrReplaceTempView("unified_payment")


In [0]:
%sql
SELECT 
    DATE(timestamp) AS date,
    payment_type,
    SUM(amount) AS total_amount
FROM unified_payment
GROUP BY 1,2
ORDER BY 1,2;

SELECT 
    customer_id,
    COUNT(*) AS num_txns,
    AVG(amount) AS avg_amount
FROM unified_payment
WHERE status = 'success'
GROUP BY customer_id
ORDER BY avg_amount DESC
LIMIT 10;

SELECT 
    DATE(timestamp) AS date,
    COUNT(*) AS failed_count
FROM unified_payment
WHERE status = 'failed'
GROUP BY 1
ORDER BY 1;

